# Tool-Calling Research Agent (Teacher)

**Phase 1** of the BBB conference demo pipeline.

This notebook implements a tool-calling agent using the **OpenAI Responses API**
with reasoning support (gpt-5.4). The agent researches a given stock ticker by
calling financial tools (yfinance), then produces a structured research memo.

The full conversation trajectory (including reasoning summaries and all tool calls)
is saved as training data for fine-tuning Qwen3-4B.

## Architecture
```
User prompt → GPT-5.4 (Responses API)
                ├── reasoning item (summary captured)
                └── function_call item → execute tool → function_call_output → loop
                                                                                 ↓
                                                                         final message
```

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Add nb/ to path so we can import bbb as a package
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_SCHEMAS, TOOL_FUNCTIONS
from bbb.agent import run_tool_calling_agent

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI(base_url="https://us.api.openai.com/v1")

MODEL = "gpt-5.4"

## System Prompt

Uses the `developer` role (Responses API equivalent of `system`).
Instructs the model to use tools for research and produce a structured memo.
Reasoning happens via the model's internal reasoning tokens — no fake `<think>` tags needed.

In [2]:
SYSTEM_PROMPT = """\
You are an equity research analyst conducting comprehensive research on a given stock.

## Instructions
- Use the available tools to gather financial data, news, price history, and analyst recommendations.
- Before each tool call, take time to think about the task at hand.
- After gathering sufficient data, synthesize your findings into a structured research memo.
- Be thorough but efficient — aim for just the necessary amount of tool calls.

## Output Format
Your final response should be a markdown research memo with these sections:
- **Company Overview**: Brief description and market position
- **Financial Analysis**: Revenue, profitability, balance sheet highlights
- **Market Performance**: Price trends, volatility, trading patterns
- **Analyst Sentiment**: Consensus recommendations, recent upgrades/downgrades
- **Key Risks & Opportunities**: Summary of bull/bear case

## Important
- Use ONLY the tools provided. Do not make up financial data.
- Present facts objectively. Do not give investment advice.
- If a tool returns an error, note it and move on.
"""

## Tool Schemas

These are defined in `tools/stock_tools.py` and shared across all BBB notebooks.

In [3]:
# Quick look at what tools the agent has access to
# Responses API uses flat format: name/description/parameters at top level
for tool in TOOL_SCHEMAS:
    params = list(tool["parameters"]["properties"].keys())
    print(f"  {tool['name']}({', '.join(params)})")
    print(f"    → {tool['description']}\n")

  get_stock_news(ticker)
    → Get the 5 most recent news articles for a stock ticker.

  get_financials(ticker, statement_type, period)
    → Get financial statements (income, balance_sheet, or cashflow) for a stock.

  get_price_history(ticker, period, interval)
    → Get historical stock price data with summary statistics.

  get_recommendations(ticker, months_back)
    → Get analyst recommendations and recent upgrades/downgrades for a stock.



## Run the Agent

Uses `client.responses.create()` with reasoning enabled.
The agent loop handles:
- Parsing `function_call` items from `response.output`
- Executing tools and sending `function_call_output` back
- Passing reasoning items between turns (required for reasoning models)
- Capturing reasoning summaries for inspection

In [ ]:
result = run_tool_calling_agent(
    client=client,
    model=MODEL,
    user_prompt="Research NVDA focusing on financial health and growth potential.",
    system_prompt=SYSTEM_PROMPT,
    reasoning_effort="medium",
)

print(f"\nTotal input items: {len(result['input'])}")
print(f"Reasoning summaries: {len(result['reasoning_summaries'])}")
print(f"Tokens — input: {result['usage']['input_tokens']}, output: {result['usage']['output_tokens']}, reasoning: {result['usage']['reasoning_tokens']}")

### Inspect the full trajectory

Everything in order: developer prompt → user → reasoning → tool calls → tool results → ... → final memo.

In [5]:
# Print the full trajectory: input items + final output items, all in order
all_items = list(result["input"]) + list(result["output"])

for i, item in enumerate(all_items):
    # Dict items (our input messages and function_call_outputs)
    if isinstance(item, dict):
        role = item.get("role", item.get("type", "?"))
        content = item.get("content", item.get("output", ""))
        if role == "developer":
            print(f"[{i}] DEVELOPER\n{str(content)[:200]}...\n")
        elif role == "user":
            print(f"[{i}] USER\n{content}\n")
        elif item.get("type") == "function_call_output":
            print(f"[{i}] TOOL RESULT (call_id: {item.get('call_id', '?')[:20]})\n{str(content)[:200]}...\n")
        else:
            print(f"[{i}] {role.upper()}\n{str(content)[:200]}...\n")
    # SDK response objects
    else:
        item_type = getattr(item, "type", "unknown")
        if item_type == "reasoning":
            summary = "(no summary)"
            if getattr(item, "summary", None) and item.summary:
                summary = item.summary[0].text
            print(f"[{i}] REASONING\n{summary}\n")
        elif item_type == "function_call":
            print(f"[{i}] FUNCTION CALL: {item.name}\n  args: {item.arguments}\n")
        elif item_type == "message":
            text = ""
            if hasattr(item, "content") and item.content:
                text = item.content[0].text
            print(f"[{i}] ASSISTANT (final)\n{text[:500]}...\n")
        else:
            print(f"[{i}] {item_type.upper()}\n{str(item)[:200]}\n")

[0] DEVELOPER
You are an equity research analyst conducting comprehensive research on a given stock.

## Instructions
- Use the available tools to gather financial data, news, price history, and analyst recommendat...

[1] USER
Research NVDA focusing on financial health and growth potential.

[2] REASONING
**Researching NVDA Financials**

I need to research NVDA efficiently using various tools. I’m considering what to prioritize for financials, like quarterly and annual reports for income, balance sheets, and cash flow. The user is interested in the company's financial health and growth potential, so I’ll need to structure it into sections. I’m thinking of using annual income and balance sheets, perhaps adding quarterly income for momentum, while parallelizing tasks like price history and news. Let’s get started with about seven tools!

[3] ASSISTANT (final)
I’ll pull the latest financial statements, price history, analyst sentiment, and recent news for NVDA so I can assess financial h

In [6]:
from IPython.display import Markdown, display

if result["output_text"]:
    display(Markdown(result["output_text"]))
else:
    print("No final text output — agent may have hit max iterations")

# NVDA Research Memo

## Company Overview
NVIDIA (NASDAQ: NVDA) is a leading semiconductor and computing platform company best known for GPUs used in AI training/inference, data centers, gaming, and accelerated computing. Its market position appears exceptionally strong: recent financials show it operating at hyperscale revenue levels with unusually high margins, reflecting strong demand for AI infrastructure and pricing power.

## Financial Analysis
### Revenue and growth
Using annual results for fiscal years ended January 31:

- **FY2026 revenue:** **$215.9B**
- **FY2025 revenue:** **$130.5B**
- **FY2024 revenue:** **$60.9B**
- **FY2023 revenue:** **$27.0B**

This implies:
- **FY2026 YoY revenue growth:** about **+65%**
- **FY2025 YoY revenue growth:** about **+114%**
- **FY2024 YoY revenue growth:** about **+126%** vs FY2023

Quarterly momentum also remained strong through FY2026:
- **Q1 FY2026:** $44.1B
- **Q2 FY2026:** $46.7B
- **Q3 FY2026:** $57.0B
- **Q4 FY2026:** $68.1B

That shows continued sequential expansion, with especially strong acceleration in the second half of the year.

### Profitability
FY2026 profitability was extremely strong:

- **Gross profit:** **$153.5B**
- **Operating income:** **$130.4B**
- **Net income:** **$120.1B**
- **Diluted EPS:** **$4.90**

Approximate FY2026 margins:
- **Gross margin:** **~71%**
- **Operating margin:** **~60%**
- **Net margin:** **~56%**

For comparison:
- FY2025 operating income was **$81.5B**
- FY2024 operating income was **$33.0B**
- FY2023 operating income was **$5.6B**

R&D spending also scaled materially:
- **FY2026 R&D:** **$18.5B**
- **FY2025 R&D:** **$12.9B**
- **FY2024 R&D:** **$8.7B**

This suggests NVIDIA is expanding aggressively while still widening margins, which is a strong sign for growth quality.

### Cash flow
- **FY2026 operating cash flow:** **$102.7B**
- **FY2026 free cash flow:** **$96.7B**
- **FY2025 free cash flow:** **$60.9B**
- **FY2024 free cash flow:** **$27.0B**

Other FY2026 cash flow highlights:
- **Capex:** **$6.0B**
- **Share repurchases:** **$40.1B**
- **Dividends paid:** **$974M**

The business is generating very large cash flows even after rising investment. That supports both internal growth spending and shareholder returns.

### Balance sheet and financial health
As of **Jan. 31, 2026**:

- **Total assets:** **$206.8B**
- **Stockholders’ equity:** **$157.3B**
- **Total debt:** **$11.0B**
- **Cash, cash equivalents, and short-term investments:** **$62.6B**
- **Working capital:** **$93.4B**

Overall, the balance sheet looks **very strong**:
- Liquidity is substantial.
- Cash and short-term investments far exceed total debt.
- Equity has expanded rapidly alongside earnings.

A few operating balance sheet items also rose sharply:
- **Inventory:** **$21.4B** vs **$10.1B** a year earlier
- **Receivables:** **$38.5B** vs **$23.1B** a year earlier

That is not surprising given the scale-up in revenue, but it does indicate growth is requiring more working capital.

## Market Performance
Over the last 1 year of weekly data:

- **Total return:** **+56.2%**
- **52-week range in dataset:** **$94.29 to $202.47**
- **Average price:** **$165.76**
- **Most recent weekly close:** **$171.24**

Observed pattern:
- Shares were highly volatile in spring 2025, bottoming near **$94**.
- They then rallied strongly into late 2025, peaking above **$200**.
- Since the peak, the stock has pulled back but remains well above the 1-year low.

Volume was consistently heavy, and especially elevated during large moves, suggesting:
- high institutional participation,
- strong sentiment sensitivity,
- and a stock that trades as both a company-specific AI leader and a macro tech proxy.

## Analyst Sentiment
Current recommendation snapshot:
- **Strong Buy:** 9
- **Buy:** 48
- **Hold:** 2
- **Sell:** 1
- **Strong Sell:** 0

That is an overwhelmingly positive consensus.

Recent actions were mostly **maintains or price-target increases**, including:
- **Raymond James:** Strong Buy, PT raised to **$323**
- **Truist:** Buy, PT raised to **$287**
- **Wedbush:** Outperform, PT raised to **$300**
- **JP Morgan:** Overweight, PT raised to **$265**
- **Citigroup:** Buy, PT raised to **$300**
- **BofA:** Buy, PT raised to **$300**

Notable dissent exists but is limited:
- **Deutsche Bank:** Hold
- **Seaport Global (Apr 2025):** Sell initiation with **$100** target

Net takeaway: Street sentiment remains strongly constructive, largely tied to continued AI demand and earnings power.

## Key Risks & Opportunities
### Opportunities
- **AI infrastructure demand:** Revenue and profit expansion strongly suggest NVIDIA remains one of the primary beneficiaries of enterprise and hyperscaler AI spending.
- **Operating leverage:** Margins have expanded dramatically as revenue scales.
- **Cash generation:** Massive free cash flow gives flexibility for R&D, supply expansion, acquisitions, and buybacks.
- **Ecosystem strength:** Recent news mentions NVIDIA among important AI/data partners, supporting its broad platform role beyond chips alone.

### Risks
- **Expectations risk:** With margins and growth already at extraordinary levels, even small deceleration could pressure sentiment.
- **Working capital build:** Inventory and receivables rose sharply; if demand normalizes, these balances may be watched closely.
- **Stock volatility:** The 1-year trading range and large weekly swings show the stock is highly sentiment-driven.
- **Sector/macro sensitivity:** Recent news flow included broad tech selloffs rather than company-specific issues, underscoring exposure to market-wide risk appetite.

## Bottom Line
From a financial health perspective, NVIDIA looks exceptionally strong: rapid revenue growth, very high profitability, huge free cash flow, and a fortress-like balance sheet. From a growth potential perspective, the recent quarterly and annual figures still show meaningful momentum, though the main debate is less about whether growth exists and more about how durable current AI-driven demand and margins prove to be.

## Save the Trajectory

We serialize the input list for training data. Responses API items
that are SDK objects get serialized via their `.model_dump()` method.

In [7]:
def serialize_input_list(input_list: list) -> list[dict]:
    """Convert a mix of dicts and SDK response objects to JSON-serializable dicts."""
    serialized = []
    for item in input_list:
        if isinstance(item, dict):
            serialized.append(item)
        elif hasattr(item, "model_dump"):
            serialized.append(item.model_dump())
        else:
            serialized.append({"type": "unknown", "data": str(item)})
    return serialized


output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "tool_calling_trajectories.jsonl"

record = {
    "input": serialize_input_list(result["input"]),
    "output": serialize_input_list(result["output"]),
    "tools": TOOL_SCHEMAS,
    "reasoning_summaries": result["reasoning_summaries"],
    "usage": result["usage"],
}

with open(output_file, "a") as f:
    f.write(json.dumps(record) + "\n")

print(f"Saved trajectory to {output_file}")

Saved trajectory to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/tool_calling_trajectories.jsonl


## Test with a Few More Tickers

In [ ]:
test_tickers = [
    ("AAPL", "competitive position and market share"),
    ("TSLA", "recent news and analyst sentiment"),
    ("JPM", "financial health and balance sheet strength"),
]

for ticker, focus in test_tickers:
    print(f"\n{'='*60}")
    print(f"Researching {ticker} — {focus}")
    print(f"{'='*60}")
    
    res = run_tool_calling_agent(
        client=client,
        model=MODEL,
        user_prompt=f"Research {ticker} focusing on {focus}.",
        system_prompt=SYSTEM_PROMPT,
        reasoning_effort="medium",
    )
    
    # Count tool calls from the input list
    n_tool_results = sum(
        1 for item in res["input"]
        if isinstance(item, dict) and item.get("type") == "function_call_output"
    )
    memo_len = len(res["output_text"]) if res["output_text"] else 0
    print(f"  -> {n_tool_results} tool calls, {len(res['reasoning_summaries'])} reasoning steps, memo: {memo_len} chars")
    print(f"  -> Tokens: {res['usage']['input_tokens']} in / {res['usage']['output_tokens']} out / {res['usage']['reasoning_tokens']} reasoning")
    
    # Save
    record = {
        "input": serialize_input_list(res["input"]),
        "output": serialize_input_list(res["output"]),
        "tools": TOOL_SCHEMAS,
        "reasoning_summaries": res["reasoning_summaries"],
        "usage": res["usage"],
    }
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")